In [1]:
# import warnings is simply used to remove warnings about incompatibility with future Pandas update on Nate's device
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
from OWF_func_run_all_functions import *
import pandas as pd
from ipynb.fs.defs.OWF_working_file import (construct_calendar_year_list, 
                                            construct_business_case_year_list, 
                                            construct_operations_years_list, 
                                            construct_decomissioning_years_list, 
                                            construction_phase,
                                            operational_phase,
                                            decommissioning_phase,
                                            taxes_and_profits_part1,
                                            debt_and_loan_part1,
                                            taxes_and_profits_part2,
                                            debt_and_loan_part2,
                                            project_reserves,
                                            equity_funding,
                                            present_value_cashflows,
                                            cumulative_equity_and_debt_cashflows,
                                            discounted_cashflows_for_levelized_cost,
                                            levelized_cost_and_revenues,
                                            project_kpi,
                                            equity_kpi,
                                            output_kpi)
from OWF_input_data import *

XMLResource loading...


In [3]:
from esdl import esdl
from esdl.esdl_handler import EnergySystemHandler

esh = EnergySystemHandler()
esh.load_file("TNVDW_cost_single_units_v1_most_likely_scenario.esdl")
energy_system = esh.get_energy_system()

instance_list = energy_system.instance
my_instance = instance_list[0]

XMLResource loading...


In [4]:
def get_input_data_from_esdl():

    #https://pyesdl.readthedocs.io/en/latest/Tutorials/tutorial4.html

    asset_types_list = []
    asset_names_list = []
    asset_ids_list = []
    asset_powers_list = []
    asset_efficiencies_list = []
    asset_investment_costs_list = [] #Capex, fixed_om and var_om are a singlevalue, not a range!
    asset_fixed_opex_list = []
    asset_var_opex_list = []
    asset_wacc_list = []


    #iterate through all ESDL elements: get all instances of type
    for esdl_element in energy_system.eAllContents():

        #check if the element is an EnergyAsset
        if isinstance(esdl_element, esdl.EnergyAsset):

            #if it is, write its type, ID and name to a corresponding list
            asset_types_list.append(esdl_element.eClass.name)
            asset_names_list.append(esdl_element.name)

            #if it has costinformation, then append
            if esdl_element.costInformation is not None:
                asset_investment_costs_list.append(esdl_element.costInformation.investmentCosts.value) #singlevalues, not a range!
                asset_fixed_opex_list.append(esdl_element.costInformation.fixedOperationalAndMaintenanceCosts.value)
                asset_var_opex_list.append(esdl_element.costInformation.variableOperationalAndMaintenanceCosts.value)
                asset_wacc_list.append(esdl_element.costInformation.discountRate.value)   
            else:
                asset_investment_costs_list.append("")
                asset_fixed_opex_list.append("")
                asset_var_opex_list.append("")
                asset_wacc_list.append("")


            #if an element is a producer, consumer or conversion, write its power
            if isinstance(esdl_element, esdl.Producer) or isinstance(esdl_element, esdl.Consumer) or isinstance(esdl_element, esdl.Conversion):
                asset_powers_list.append(esdl_element.power)
            else:
                asset_powers_list.append("")

            #if an element is an electrolyzer, write its efficiency
            if isinstance(esdl_element, esdl.Electrolyzer):
                asset_efficiencies_list.append(esdl_element.efficiency)
            else:
                asset_efficiencies_list.append("")


    # maybe include some units!

    #create empty dataframe
    asset_parameters = pd.DataFrame(index=asset_names_list)
    asset_parameters.columns.name = 'name'

    # fill in the data in the dataframe
    asset_parameters["type"] = asset_types_list                   # e.g., Windpark, Electrolyser, GasDemand, ElectricityCable, Pipe
    asset_parameters["power"] = asset_powers_list
    asset_parameters["efficiency"] = asset_efficiencies_list
    asset_parameters["investment_costs"] = asset_investment_costs_list
    asset_parameters["fixed_opex"] = asset_fixed_opex_list
    asset_parameters["variable_opex"] = asset_var_opex_list
    asset_parameters["wacc"] = asset_wacc_list

    #decided to remove these from dataframe
    asset_parameters = asset_parameters.drop("type", axis=1)
    asset_parameters = asset_parameters.drop("ElectricityCable", axis=0)
    asset_parameters = asset_parameters.drop("H2-pipe", axis=0)

    #display dataframe 
    return asset_parameters

In [5]:
get_input_data_from_esdl()

name,power,efficiency,investment_costs,fixed_opex,variable_opex,wacc
TNVDW,700000000.0,,1750.0,2.25,5.0,8.5
Electrolyzer,500.0,0.6,2000.0,2.0,0.0,8.25
Offtaker,24900000.0,,0.0,0.0,0.0,10.5


In [6]:
def test():

    esh = EnergySystemHandler()
    esh.load_file("TNVDW_cost_single_units_v1_pessimistic_scenario.esdl")
    energy_system = esh.get_energy_system()
    instance_list = energy_system.instance
    my_instance = instance_list[0]
    
    print(get_input_data_from_esdl())


    esh = EnergySystemHandler()
    esh.load_file("TNVDW_cost_single_units_v1_most_likely_scenario.esdl")
    energy_system = esh.get_energy_system()
    instance_list = energy_system.instance
    my_instance = instance_list[0]
    
    print(get_input_data_from_esdl())

In [7]:
file_names = ["TNVDW_cost_single_units_v1_pessimistic_scenario.esdl", "TNVDW_cost_single_units_v1_most_likely_scenario.esdl", "TNVDW_cost_single_units_v1_optimistic_scenario.esdl"]

for x in file_names:

    esh = EnergySystemHandler()
    esh.load_file(x)
    energy_system = esh.get_energy_system()
    instance_list = energy_system.instance
    my_instance = instance_list[0]

    print(get_input_data_from_esdl())

    if x == file_names[0]:
        drop_scenarios = ['most_likely','optimistic']
        print(drop_scenarios)
        print(get_input_data_from_esdl())

    elif x == file_names[1]:
        drop_scenarios = ['pessimistic','optimistic']
        print(drop_scenarios)
        print(get_input_data_from_esdl())

    else: 
        drop_scenarios = ['pessimistic','most_likely']
        print(drop_scenarios)
        print(get_input_data_from_esdl())


    ###################### HERE ARE ALL THE INPUTS THAT DEPEND ON THE SCENARIO AGAIN  ################################

    ########## Get cost data from Mapeditor

    # owf_capex = asset_parameters['investment_costs']['TNVDW']         # in MEUR
    # owf_fixed_opex = asset_parameters['fixed_opex']['TNVDW']/100      # converted 2 percent to 0.02
    # owf_var_opex = asset_parameters['variable_opex']['TNVDW']         # in Eur/MWh
    owf_general_WACC = asset_parameters['wacc']['TNVDW']/100          # from % to decimal

    
    ########## Cost data from factsheet

    # Used factsheet: NSE5_Factsheet_OffshoreWind 
    # Date: 14-08-2024

    # format: [2030[high-mid-low], 2040[high-mid-low], 2050[high-mid-low]]

    windfarm_capacity = 700                                                         # MW, based on TNVDW
    windturbine_capacity = [[15,17,19],[18,21,25],[18,21,25]]                       # MW 

    capex_rna = [[16,12.3,9],[16,13.8,9],[16,13.8,9]]                               # MEUR/WT
    capex_structure = [[10,8.3,6],[10,13.19,6],[10,13.9,6]]                         # MEUR/WT
    capex_electric = [[1500,693,800],[1500,693,800],[1500,693,800]]                 # kEur/MW
    capex_cables = [[2.112,2.112,2.112],[1.816,1.816,1.816],[1.816,1.816,1.816]]    # kEUR/MW/km
    capex_installation = [[200,245,100],[200,211,100],[200,211,100]]                # kEUR/MW
    capex_project_costs = [[200,313,100],[200,314,100],[200,314,100]]               # kEUR/MW
    capex_abex_decex = [[160,211,80],[160,196,80],[160,196,80]]                     # kEUR/MW abandonment/decomissioning costs
    opex = [[70,52,30],[70,37,30],[70,37,30]]                                       # kEUR/MW/yr

    factsheet_parameters = [windturbine_capacity, capex_rna, capex_structure, capex_electric, capex_cables, capex_installation, capex_project_costs, capex_abex_decex, opex]
    index_names = ['windturbine_capacity', 'capex_rna', 'capex_structure', 'capex_electric', 'capex_cables', 'capex_installation', 'capex_project_costs', 'capex_abex_decex', 'opex',]

    df_factsheet_2030 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])
    df_factsheet_2040 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])
    df_factsheet_2050 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])

    for i,name in enumerate(index_names):
        df_factsheet_2030.loc[name] = factsheet_parameters[i][0]

    for i,name in enumerate(index_names):
        df_factsheet_2040.loc[name] = factsheet_parameters[i][1]

    for i,name in enumerate(index_names):
        df_factsheet_2050.loc[name] = factsheet_parameters[i][2]

    df_factsheet_2030.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_factsheet_2040.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_factsheet_2050.drop(columns=drop_scenarios,axis=1,inplace=True)

    df_factsheet_2030_2050 = pd.concat([df_factsheet_2030,df_factsheet_2040,df_factsheet_2050], axis=1)
    df_factsheet_2030_2050.columns = ['2030','2040','2050']

    df_factsheet_2030_2050.style.format(precision=2)


    ########## Transform 2030, 2040, 2050 data from factsheet into yearly data

    all_years = range(2027,2071,1)
    df_yearly_data = pd.DataFrame(index=index_names, columns=all_years)

    for year in all_years:
        for name in index_names:
            if year <= 2030:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2030'][name]

    for i, year in enumerate(all_years):
        for name in index_names:
            if year >2030 and year < 2040:
                df_yearly_data[year][name] = (df_factsheet_2030_2050['2030'][name] + 
                                            ((df_factsheet_2030_2050['2040'][name] - 
                                            df_factsheet_2030_2050['2030'][name]) / 10)*(i-3))
                
    for i, year in enumerate(all_years):
        for name in index_names:
            if year == 2040:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2040'][name]
            if year >2040 and year < 2050:
                df_yearly_data[year][name] = (df_factsheet_2030_2050['2040'][name] + 
                                            ((df_factsheet_2030_2050['2050'][name] - 
                                            df_factsheet_2030_2050['2040'][name]) / 10)*(i-13))

    for year in all_years:
        for name in index_names:
            if year >= 2050:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2050'][name]

    df_yearly_data.style.format(precision=1)


    # here we make a df with just the opex numbers, so we can use this for the sensitivity later on
    df_owf_opex = pd.DataFrame(columns=all_years)
    df_owf_opex.loc['opex'] = np.array(df_yearly_data.loc['opex'])
    df_owf_opex


    ########## Input data from Excel

    # Data format: [pessimistic, most_likely, optimistic]

    # General data: business case length
    duration_construction_list = [3, 3, 3]
    duration_operation_list = [25, 25, 30]
    duration_decommissioning_list = [2, 2, 2]
    tender_year_list = [2027, 2027, 2027]                       # year at which business case is 0, 
                                                                # i.e. year before start construction

    # Cost data from 'inflation-WACC' sheet
    owf_income_tax_rate_list = [0.258, 0.258, 0.258]            # in %
    owf_inflation_list = [0.02, 0.02, 0.02]                     # in %
    # owf_general_WACC_list = [0.0925, 0.085, 0.0775]             # in %
    owf_loan_interest_rate_list = [0.065, 0.05, 0.035]          # in %
    owf_length_of_loan_list = [15, 15, 15]                      # years
    owf_loan_type = 'annuity'
    owf_depreciation_list = [25, 25, 30]                        # years

    # Cost data from 'cost data OWF' sheet
    owf_contingency_list = [0.1, 0.1, 0.1]                      # in %
    owf_loan_percentage_list = [0.75, 0.75, 0.75]               # in %
    owf_decomissioning_percentage_list = [0.02, 0.02, 0.02]     # in %

    # Cost data from 'PPA OWF' sheet
    # In 'used input data' Excel file this is linked to the draft data input EYE document
    owf_sold_to_electrolyser_list = [2849543, 2849543, 2849543]     # MWh/year 
    owf_sold_to_grid_list = [195458, 195458, 195458]                # MWh/year 
    owf_revenues_to_electrolyser_list = [119.12, 158.83, 198.54]    # MEUR 
    owf_revenues_to_market_list = [3.89, 5.19, 6.48]                # MEUR 


    # Construct a dataframe with the input from above
    df_cost_data = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])

    df_cost_data.loc['duration_construction'] = duration_construction_list
    df_cost_data.loc['duration_operation'] = duration_operation_list
    df_cost_data.loc['duration_decommissioning'] = duration_decommissioning_list
    df_cost_data.loc['tender_year'] = tender_year_list

    df_cost_data.loc['income_tax_rate'] = owf_income_tax_rate_list
    df_cost_data.loc['inflation'] = owf_inflation_list
    # df_cost_data.loc['wacc'] = owf_general_WACC_list
    df_cost_data.loc['loan_interest_rate'] = owf_loan_interest_rate_list
    df_cost_data.loc['length_of_loan'] = owf_length_of_loan_list
    df_cost_data.loc['depreciation'] = owf_depreciation_list

    df_cost_data.loc['contingency'] = owf_contingency_list
    df_cost_data.loc['loan_percentage'] = owf_loan_percentage_list
    df_cost_data.loc['decommissioning_percentage'] = owf_decomissioning_percentage_list

    df_cost_data.loc['sold_to_electrolyser'] = owf_sold_to_electrolyser_list
    df_cost_data.loc['sold_to_grid'] = owf_sold_to_grid_list
    df_cost_data.loc['revenues_to_electrolyser'] = owf_revenues_to_electrolyser_list
    df_cost_data.loc['revenues_to_market'] = owf_revenues_to_market_list

    df_cost_data

    # the scenario is chosen, by dropping the other two scenario's from the dataframe

    df_cost_data.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_cost_data.style.format(precision=3)

    # get cost data for the correct scenario
    # in the df we can see that we have the most_likely scenario
    # we can get the parameters from the df by using the row index
    # using df_cost_data.loc['parameter'] not only gives the value, but also column name, dtype etc.
    # therefore, we use .item() to get the desired value

    duration_construction = int(df_cost_data.loc['duration_construction'].item())
    duration_operation = int(df_cost_data.loc['duration_operation'].item())
    duration_decommissioning = int(df_cost_data.loc['duration_decommissioning'].item())
    tender_year = int(df_cost_data.loc['tender_year'].item())

    owf_income_tax_rate = df_cost_data.loc['income_tax_rate'].item()
    owf_inflation = df_cost_data.loc['inflation'].item()                 
    # owf_general_WACC = df_cost_data.loc['wacc'].item()
    owf_loan_interest_rate = df_cost_data.loc['loan_interest_rate'].item()
    owf_length_of_loan = int(df_cost_data.loc['length_of_loan'].item())
    owf_depreciation = int(df_cost_data.loc['depreciation'].item())

    owf_contingency = df_cost_data.loc['contingency'].item()
    owf_loan_percentage = df_cost_data.loc['loan_percentage'].item()
    owf_decomissioning_percentage = df_cost_data.loc['decommissioning_percentage'].item()

    owf_sold_to_electrolyser = df_cost_data.loc['sold_to_electrolyser'].item()
    owf_sold_to_grid = df_cost_data.loc['sold_to_grid'].item()
    owf_revenues_to_electrolyser = df_cost_data.loc['revenues_to_electrolyser'].item()
    owf_revenues_to_market = df_cost_data.loc['revenues_to_market'].item()


    ########## Calculate cost data

    # We need the number of turbines because some of the cost data is given per turbine
    # We assume that at least 700 MW windfarm capacity is required
    # Therefore we use np.ceil (or you could use math.ceil) to round the value up to the closest integer
    # Then we multiply the rounded number with the turbine capacity to get a new total windfarm capacity,
    # which is equal to or slightly higher than the original 700 MW capacity

    number_of_turbines = np.ceil(windfarm_capacity / df_yearly_data[tender_year]['windturbine_capacity'])
    new_windfarm_capaxity = number_of_turbines * df_yearly_data[tender_year]['windturbine_capacity']

    # The cable costs depend on the length of the cables
    # In Mapeditor, a straight line between OWF and Eemshaven is approximately 111694 meter
    cable_distance = 111694 / 1000     # km

    owf_capex = (df_yearly_data[tender_year]['capex_rna'] * number_of_turbines +
                    df_yearly_data[tender_year]['capex_structure'] * number_of_turbines +
                    df_yearly_data[tender_year]['capex_electric'] * new_windfarm_capaxity / 1000 +
                    df_yearly_data[tender_year]['capex_cables'] * new_windfarm_capaxity / 1000 * cable_distance +
                    df_yearly_data[tender_year]['capex_installation'] * new_windfarm_capaxity / 1000 +
                    df_yearly_data[tender_year]['capex_project_costs'] * new_windfarm_capaxity / 1000)




    ##################################### END OF ALL THE INPUTS THAT DEPEND ON THE SCENARIO    ################################


    owf_run_all_functions(owf_capex,
                        df_owf_opex,
                        owf_inflation,
                        owf_revenues_to_electrolyser,
                        owf_revenues_to_market,
                        owf_loan_percentage,
                        owf_loan_interest_rate,
                        owf_income_tax_rate,
                        owf_general_WACC,
                        duration_operation)
        
    print(levelized_cost_and_revenues(owf_capex,
                                    df_owf_opex,
                                    owf_inflation,
                                    owf_revenues_to_electrolyser,
                                    owf_revenues_to_market,
                                    owf_loan_percentage,
                                    owf_loan_interest_rate,
                                    owf_income_tax_rate,
                                    owf_general_WACC,
                                    duration_operation))

XMLResource loading...
name                power efficiency investment_costs fixed_opex  \
TNVDW         700000000.0                      2450.0        3.0   
Electrolyzer        500.0        0.6           2500.0        2.0   
Offtaker       24900000.0                         0.0        0.0   

name         variable_opex  wacc  
TNVDW                  6.0  9.25  
Electrolyzer           0.0   9.5  
Offtaker               0.0  11.3  
['most_likely', 'optimistic']
name                power efficiency investment_costs fixed_opex  \
TNVDW         700000000.0                      2450.0        3.0   
Electrolyzer        500.0        0.6           2500.0        2.0   
Offtaker       24900000.0                         0.0        0.0   

name         variable_opex  wacc  
TNVDW                  6.0  9.25  
Electrolyzer           0.0   9.5  
Offtaker               0.0  11.3  


AttributeError: 'float' object has no attribute 'loc'

In [ ]:
file_names = ["TNVDW_cost_single_units_v1_pessimistic_scenario.esdl", "TNVDW_cost_single_units_v1_most_likely_scenario.esdl", "TNVDW_cost_single_units_v1_optimistic_scenario.esdl"]

for x in file_names:

    esh = EnergySystemHandler()
    esh.load_file(x)
    energy_system = esh.get_energy_system()
    instance_list = energy_system.instance
    my_instance = instance_list[0]

    print(get_input_data_from_esdl())

    if x == file_names[0]:
        drop_scenarios = ['most_likely','optimistic']
        print(drop_scenarios)
        print(get_input_data_from_esdl())

    elif x == file_names[1]:
        drop_scenarios = ['pessimistic','optimistic']
        print(drop_scenarios)
        print(get_input_data_from_esdl())

    else: 
        drop_scenarios = ['pessimistic','most_likely']
        print(drop_scenarios)
        print(get_input_data_from_esdl())


    ###################### HERE ARE ALL THE INPUTS THAT DEPEND ON THE SCENARIO AGAIN  ################################

    ########## Get cost data from Mapeditor

    # owf_capex = asset_parameters['investment_costs']['TNVDW']         # in MEUR
    # owf_fixed_opex = asset_parameters['fixed_opex']['TNVDW']/100      # converted 2 percent to 0.02
    # owf_var_opex = asset_parameters['variable_opex']['TNVDW']         # in Eur/MWh
    owf_general_WACC = asset_parameters['wacc']['TNVDW']/100          # from % to decimal

    
    ########## Cost data from factsheet

    # Used factsheet: NSE5_Factsheet_OffshoreWind 
    # Date: 14-08-2024

    # format: [2030[high-mid-low], 2040[high-mid-low], 2050[high-mid-low]]

    windfarm_capacity = 700                                                         # MW, based on TNVDW
    windturbine_capacity = [[15,17,19],[18,21,25],[18,21,25]]                       # MW 

    capex_rna = [[16,12.3,9],[16,13.8,9],[16,13.8,9]]                               # MEUR/WT
    capex_structure = [[10,8.3,6],[10,13.19,6],[10,13.9,6]]                         # MEUR/WT
    capex_electric = [[1500,693,800],[1500,693,800],[1500,693,800]]                 # kEur/MW
    capex_cables = [[2.112,2.112,2.112],[1.816,1.816,1.816],[1.816,1.816,1.816]]    # kEUR/MW/km
    capex_installation = [[200,245,100],[200,211,100],[200,211,100]]                # kEUR/MW
    capex_project_costs = [[200,313,100],[200,314,100],[200,314,100]]               # kEUR/MW
    capex_abex_decex = [[160,211,80],[160,196,80],[160,196,80]]                     # kEUR/MW abandonment/decomissioning costs
    opex = [[70,52,30],[70,37,30],[70,37,30]]                                       # kEUR/MW/yr

    factsheet_parameters = [windturbine_capacity, capex_rna, capex_structure, capex_electric, capex_cables, capex_installation, capex_project_costs, capex_abex_decex, opex]
    index_names = ['windturbine_capacity', 'capex_rna', 'capex_structure', 'capex_electric', 'capex_cables', 'capex_installation', 'capex_project_costs', 'capex_abex_decex', 'opex',]

    df_factsheet_2030 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])
    df_factsheet_2040 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])
    df_factsheet_2050 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])

    for i,name in enumerate(index_names):
        df_factsheet_2030.loc[name] = factsheet_parameters[i][0]

    for i,name in enumerate(index_names):
        df_factsheet_2040.loc[name] = factsheet_parameters[i][1]

    for i,name in enumerate(index_names):
        df_factsheet_2050.loc[name] = factsheet_parameters[i][2]

    df_factsheet_2030.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_factsheet_2040.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_factsheet_2050.drop(columns=drop_scenarios,axis=1,inplace=True)

    df_factsheet_2030_2050 = pd.concat([df_factsheet_2030,df_factsheet_2040,df_factsheet_2050], axis=1)
    df_factsheet_2030_2050.columns = ['2030','2040','2050']

    df_factsheet_2030_2050.style.format(precision=2)


    ########## Transform 2030, 2040, 2050 data from factsheet into yearly data

    all_years = range(2027,2071,1)
    df_yearly_data = pd.DataFrame(index=index_names, columns=all_years)

    for year in all_years:
        for name in index_names:
            if year <= 2030:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2030'][name]

    for i, year in enumerate(all_years):
        for name in index_names:
            if year >2030 and year < 2040:
                df_yearly_data[year][name] = (df_factsheet_2030_2050['2030'][name] + 
                                            ((df_factsheet_2030_2050['2040'][name] - 
                                            df_factsheet_2030_2050['2030'][name]) / 10)*(i-3))
                
    for i, year in enumerate(all_years):
        for name in index_names:
            if year == 2040:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2040'][name]
            if year >2040 and year < 2050:
                df_yearly_data[year][name] = (df_factsheet_2030_2050['2040'][name] + 
                                            ((df_factsheet_2030_2050['2050'][name] - 
                                            df_factsheet_2030_2050['2040'][name]) / 10)*(i-13))

    for year in all_years:
        for name in index_names:
            if year >= 2050:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2050'][name]

    df_yearly_data.style.format(precision=1)


    # here we make a df with just the opex numbers, so we can use this for the sensitivity later on
    df_owf_opex = pd.DataFrame(columns=all_years)
    df_owf_opex.loc['opex'] = np.array(df_yearly_data.loc['opex'])
    df_owf_opex


    ########## Input data from Excel

    # Data format: [pessimistic, most_likely, optimistic]

    # General data: business case length
    duration_construction_list = [3, 3, 3]
    duration_operation_list = [25, 25, 30]
    duration_decommissioning_list = [2, 2, 2]
    tender_year_list = [2027, 2027, 2027]                       # year at which business case is 0, 
                                                                # i.e. year before start construction

    # Cost data from 'inflation-WACC' sheet
    owf_income_tax_rate_list = [0.258, 0.258, 0.258]            # in %
    owf_inflation_list = [0.02, 0.02, 0.02]                     # in %
    # owf_general_WACC_list = [0.0925, 0.085, 0.0775]             # in %
    owf_loan_interest_rate_list = [0.065, 0.05, 0.035]          # in %
    owf_length_of_loan_list = [15, 15, 15]                      # years
    owf_loan_type = 'annuity'
    owf_depreciation_list = [25, 25, 30]                        # years

    # Cost data from 'cost data OWF' sheet
    owf_contingency_list = [0.1, 0.1, 0.1]                      # in %
    owf_loan_percentage_list = [0.75, 0.75, 0.75]               # in %
    owf_decomissioning_percentage_list = [0.02, 0.02, 0.02]     # in %

    # Cost data from 'PPA OWF' sheet
    # In 'used input data' Excel file this is linked to the draft data input EYE document
    owf_sold_to_electrolyser_list = [2849543, 2849543, 2849543]     # MWh/year 
    owf_sold_to_grid_list = [195458, 195458, 195458]                # MWh/year 
    owf_revenues_to_electrolyser_list = [119.12, 158.83, 198.54]    # MEUR 
    owf_revenues_to_market_list = [3.89, 5.19, 6.48]                # MEUR 


    # Construct a dataframe with the input from above
    df_cost_data = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])

    df_cost_data.loc['duration_construction'] = duration_construction_list
    df_cost_data.loc['duration_operation'] = duration_operation_list
    df_cost_data.loc['duration_decommissioning'] = duration_decommissioning_list
    df_cost_data.loc['tender_year'] = tender_year_list

    df_cost_data.loc['income_tax_rate'] = owf_income_tax_rate_list
    df_cost_data.loc['inflation'] = owf_inflation_list
    # df_cost_data.loc['wacc'] = owf_general_WACC_list
    df_cost_data.loc['loan_interest_rate'] = owf_loan_interest_rate_list
    df_cost_data.loc['length_of_loan'] = owf_length_of_loan_list
    df_cost_data.loc['depreciation'] = owf_depreciation_list

    df_cost_data.loc['contingency'] = owf_contingency_list
    df_cost_data.loc['loan_percentage'] = owf_loan_percentage_list
    df_cost_data.loc['decommissioning_percentage'] = owf_decomissioning_percentage_list

    df_cost_data.loc['sold_to_electrolyser'] = owf_sold_to_electrolyser_list
    df_cost_data.loc['sold_to_grid'] = owf_sold_to_grid_list
    df_cost_data.loc['revenues_to_electrolyser'] = owf_revenues_to_electrolyser_list
    df_cost_data.loc['revenues_to_market'] = owf_revenues_to_market_list

    df_cost_data

    # the scenario is chosen, by dropping the other two scenario's from the dataframe

    df_cost_data.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_cost_data.style.format(precision=3)

    # get cost data for the correct scenario
    # in the df we can see that we have the most_likely scenario
    # we can get the parameters from the df by using the row index
    # using df_cost_data.loc['parameter'] not only gives the value, but also column name, dtype etc.
    # therefore, we use .item() to get the desired value

    duration_construction = int(df_cost_data.loc['duration_construction'].item())
    duration_operation = int(df_cost_data.loc['duration_operation'].item())
    duration_decommissioning = int(df_cost_data.loc['duration_decommissioning'].item())
    tender_year = int(df_cost_data.loc['tender_year'].item())

    owf_income_tax_rate = df_cost_data.loc['income_tax_rate'].item()
    owf_inflation = df_cost_data.loc['inflation'].item()                 
    # owf_general_WACC = df_cost_data.loc['wacc'].item()
    owf_loan_interest_rate = df_cost_data.loc['loan_interest_rate'].item()
    owf_length_of_loan = int(df_cost_data.loc['length_of_loan'].item())
    owf_depreciation = int(df_cost_data.loc['depreciation'].item())

    owf_contingency = df_cost_data.loc['contingency'].item()
    owf_loan_percentage = df_cost_data.loc['loan_percentage'].item()
    owf_decomissioning_percentage = df_cost_data.loc['decommissioning_percentage'].item()

    owf_sold_to_electrolyser = df_cost_data.loc['sold_to_electrolyser'].item()
    owf_sold_to_grid = df_cost_data.loc['sold_to_grid'].item()
    owf_revenues_to_electrolyser = df_cost_data.loc['revenues_to_electrolyser'].item()
    owf_revenues_to_market = df_cost_data.loc['revenues_to_market'].item()


    ########## Calculate cost data

    # We need the number of turbines because some of the cost data is given per turbine
    # We assume that at least 700 MW windfarm capacity is required
    # Therefore we use np.ceil (or you could use math.ceil) to round the value up to the closest integer
    # Then we multiply the rounded number with the turbine capacity to get a new total windfarm capacity,
    # which is equal to or slightly higher than the original 700 MW capacity

    number_of_turbines = np.ceil(windfarm_capacity / df_yearly_data[tender_year]['windturbine_capacity'])
    new_windfarm_capaxity = number_of_turbines * df_yearly_data[tender_year]['windturbine_capacity']

    # The cable costs depend on the length of the cables
    # In Mapeditor, a straight line between OWF and Eemshaven is approximately 111694 meter
    cable_distance = 111694 / 1000     # km

    owf_capex = (df_yearly_data[tender_year]['capex_rna'] * number_of_turbines +
                    df_yearly_data[tender_year]['capex_structure'] * number_of_turbines +
                    df_yearly_data[tender_year]['capex_electric'] * new_windfarm_capaxity / 1000 +
                    df_yearly_data[tender_year]['capex_cables'] * new_windfarm_capaxity / 1000 * cable_distance +
                    df_yearly_data[tender_year]['capex_installation'] * new_windfarm_capaxity / 1000 +
                    df_yearly_data[tender_year]['capex_project_costs'] * new_windfarm_capaxity / 1000)




    ##################################### END OF ALL THE INPUTS THAT DEPEND ON THE SCENARIO    ################################


    owf_run_all_functions(owf_capex,
                        df_owf_opex,
                        owf_inflation,
                        owf_revenues_to_electrolyser,
                        owf_revenues_to_market,
                        owf_loan_percentage,
                        owf_loan_interest_rate,
                        owf_income_tax_rate,
                        owf_general_WACC,
                        duration_operation)
        
    print(levelized_cost_and_revenues(owf_capex,
                                    df_owf_opex,
                                    owf_inflation,
                                    owf_revenues_to_electrolyser,
                                    owf_revenues_to_market,
                                    owf_loan_percentage,
                                    owf_loan_interest_rate,
                                    owf_income_tax_rate,
                                    owf_general_WACC,
                                    duration_operation))

XMLResource loading...
name                power efficiency investment_costs fixed_opex  \
TNVDW         700000000.0                      2450.0        3.0   
Electrolyzer        500.0        0.6           2500.0        2.0   
Offtaker       24900000.0                                          

name         variable_opex  wacc  
TNVDW                  6.0  9.25  
Electrolyzer           0.0   9.5  
Offtaker                          
['most_likely', 'optimistic']
name                power efficiency investment_costs fixed_opex  \
TNVDW         700000000.0                      2450.0        3.0   
Electrolyzer        500.0        0.6           2500.0        2.0   
Offtaker       24900000.0                                          

name         variable_opex  wacc  
TNVDW                  6.0  9.25  
Electrolyzer           0.0   9.5  
Offtaker                          
levelized cost calculations       cost    revenues
capex                        95.184337    0.000000
opex               

In [ ]:


file_names = ["TNVDW_cost_single_units_v1_pessimistic_scenario.esdl", "TNVDW_cost_single_units_v1_most_likely_scenario.esdl", "TNVDW_cost_single_units_v1_optimistic_scenario.esdl"]

# make a dataframe that will contain the levelized costs and revenues from all scenarios    
# Define the levels of the MultiIndex
groups = ['pessimistic', 'most_likely', 'optimistic']  # Group names
sub_columns = ['costs', 'revenues']  # Sub-column names

# Create the MultiIndex from the groups and sub_columns
multi_index = pd.MultiIndex.from_product([groups, sub_columns], names=['Scenario', 'Type'])

# Create an empty DataFrame with the multi-level columns
df_lcr_all_scenarios = pd.DataFrame(columns=multi_index)


for x in file_names:

    esh = EnergySystemHandler()
    esh.load_file(x)
    energy_system = esh.get_energy_system()
    instance_list = energy_system.instance
    my_instance = instance_list[0]

    if x == file_names[0]:
        drop_scenarios = ['most_likely','optimistic']

    elif x == file_names[1]:
        drop_scenarios = ['pessimistic','optimistic']

    else: 
        drop_scenarios = ['pessimistic','most_likely']


    ###################### HERE ARE ALL THE INPUTS THAT DEPEND ON THE SCENARIO AGAIN  ################################

    ########## Get cost data from Mapeditor

    # owf_capex = asset_parameters['investment_costs']['TNVDW']         # in MEUR
    # owf_fixed_opex = asset_parameters['fixed_opex']['TNVDW']/100      # converted 2 percent to 0.02
    # owf_var_opex = asset_parameters['variable_opex']['TNVDW']         # in Eur/MWh
    owf_general_WACC = asset_parameters['wacc']['TNVDW']/100          # from % to decimal

    
    ########## Cost data from factsheet

    # Used factsheet: NSE5_Factsheet_OffshoreWind 
    # Date: 14-08-2024

    # format: [2030[high-mid-low], 2040[high-mid-low], 2050[high-mid-low]]

    windfarm_capacity = 700                                                         # MW, based on TNVDW
    windturbine_capacity = [[15,17,19],[18,21,25],[18,21,25]]                       # MW 

    capex_rna = [[16,12.3,9],[16,13.8,9],[16,13.8,9]]                               # MEUR/WT
    capex_structure = [[10,8.3,6],[10,13.19,6],[10,13.9,6]]                         # MEUR/WT
    capex_electric = [[1500,693,800],[1500,693,800],[1500,693,800]]                 # kEur/MW
    capex_cables = [[2.112,2.112,2.112],[1.816,1.816,1.816],[1.816,1.816,1.816]]    # kEUR/MW/km
    capex_installation = [[200,245,100],[200,211,100],[200,211,100]]                # kEUR/MW
    capex_project_costs = [[200,313,100],[200,314,100],[200,314,100]]               # kEUR/MW
    capex_abex_decex = [[160,211,80],[160,196,80],[160,196,80]]                     # kEUR/MW abandonment/decomissioning costs
    opex = [[70,52,30],[70,37,30],[70,37,30]]                                       # kEUR/MW/yr

    factsheet_parameters = [windturbine_capacity, capex_rna, capex_structure, capex_electric, capex_cables, capex_installation, capex_project_costs, capex_abex_decex, opex]
    index_names = ['windturbine_capacity', 'capex_rna', 'capex_structure', 'capex_electric', 'capex_cables', 'capex_installation', 'capex_project_costs', 'capex_abex_decex', 'opex',]

    df_factsheet_2030 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])
    df_factsheet_2040 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])
    df_factsheet_2050 = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])

    for i,name in enumerate(index_names):
        df_factsheet_2030.loc[name] = factsheet_parameters[i][0]

    for i,name in enumerate(index_names):
        df_factsheet_2040.loc[name] = factsheet_parameters[i][1]

    for i,name in enumerate(index_names):
        df_factsheet_2050.loc[name] = factsheet_parameters[i][2]

    df_factsheet_2030.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_factsheet_2040.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_factsheet_2050.drop(columns=drop_scenarios,axis=1,inplace=True)

    df_factsheet_2030_2050 = pd.concat([df_factsheet_2030,df_factsheet_2040,df_factsheet_2050], axis=1)
    df_factsheet_2030_2050.columns = ['2030','2040','2050']

    df_factsheet_2030_2050.style.format(precision=2)


    ########## Transform 2030, 2040, 2050 data from factsheet into yearly data

    all_years = range(2027,2071,1)
    df_yearly_data = pd.DataFrame(index=index_names, columns=all_years)

    for year in all_years:
        for name in index_names:
            if year <= 2030:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2030'][name]
    for i, year in enumerate(all_years):
        for name in index_names:
            if year >2030 and year < 2040:
                df_yearly_data[year][name] = (df_factsheet_2030_2050['2030'][name] + 
                                            ((df_factsheet_2030_2050['2040'][name] - 
                                            df_factsheet_2030_2050['2030'][name]) / 10)*(i-3))
                
    for i, year in enumerate(all_years):
        for name in index_names:
            if year == 2040:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2040'][name]
            if year >2040 and year < 2050:
                df_yearly_data[year][name] = (df_factsheet_2030_2050['2040'][name] + 
                                            ((df_factsheet_2030_2050['2050'][name] - 
                                            df_factsheet_2030_2050['2040'][name]) / 10)*(i-13))

    for year in all_years:
        for name in index_names:
            if year >= 2050:
                df_yearly_data[year][name] = df_factsheet_2030_2050['2050'][name]

    df_yearly_data.style.format(precision=1)


    # here we make a df with just the opex numbers, so we can use this for the sensitivity later on
    df_owf_opex = pd.DataFrame(columns=all_years)
    df_owf_opex.loc['opex'] = np.array(df_yearly_data.loc['opex'])
    df_owf_opex


    ########## Input data from Excel

    # Data format: [pessimistic, most_likely, optimistic]

    # General data: business case length
    duration_construction_list = [3, 3, 3]
    duration_operation_list = [25, 25, 30]
    duration_decommissioning_list = [2, 2, 2]
    tender_year_list = [2027, 2027, 2027]                       # year at which business case is 0, 
                                                                # i.e. year before start construction

    # Cost data from 'inflation-WACC' sheet
    owf_income_tax_rate_list = [0.258, 0.258, 0.258]            # in %
    owf_inflation_list = [0.02, 0.02, 0.02]                     # in %
    # owf_general_WACC_list = [0.0925, 0.085, 0.0775]             # in %
    owf_loan_interest_rate_list = [0.065, 0.05, 0.035]          # in %
    owf_length_of_loan_list = [15, 15, 15]                      # years
    owf_loan_type = 'annuity'
    owf_depreciation_list = [25, 25, 30]                        # years

    # Cost data from 'cost data OWF' sheet
    owf_contingency_list = [0.1, 0.1, 0.1]                      # in %
    owf_loan_percentage_list = [0.75, 0.75, 0.75]               # in %
    owf_decomissioning_percentage_list = [0.02, 0.02, 0.02]     # in %

    # Cost data from 'PPA OWF' sheet
    # In 'used input data' Excel file this is linked to the draft data input EYE document
    owf_sold_to_electrolyser_list = [2849543, 2849543, 2849543]     # MWh/year 
    owf_sold_to_grid_list = [195458, 195458, 195458]                # MWh/year 
    owf_revenues_to_electrolyser_list = [119.12, 158.83, 198.54]    # MEUR 
    owf_revenues_to_market_list = [3.89, 5.19, 6.48]                # MEUR 


    # Construct a dataframe with the input from above
    df_cost_data = pd.DataFrame(columns=['pessimistic','most_likely','optimistic'])

    df_cost_data.loc['duration_construction'] = duration_construction_list
    df_cost_data.loc['duration_operation'] = duration_operation_list
    df_cost_data.loc['duration_decommissioning'] = duration_decommissioning_list
    df_cost_data.loc['tender_year'] = tender_year_list

    df_cost_data.loc['income_tax_rate'] = owf_income_tax_rate_list
    df_cost_data.loc['inflation'] = owf_inflation_list
    # df_cost_data.loc['wacc'] = owf_general_WACC_list
    df_cost_data.loc['loan_interest_rate'] = owf_loan_interest_rate_list
    df_cost_data.loc['length_of_loan'] = owf_length_of_loan_list
    df_cost_data.loc['depreciation'] = owf_depreciation_list

    df_cost_data.loc['contingency'] = owf_contingency_list
    df_cost_data.loc['loan_percentage'] = owf_loan_percentage_list
    df_cost_data.loc['decommissioning_percentage'] = owf_decomissioning_percentage_list

    df_cost_data.loc['sold_to_electrolyser'] = owf_sold_to_electrolyser_list
    df_cost_data.loc['sold_to_grid'] = owf_sold_to_grid_list
    df_cost_data.loc['revenues_to_electrolyser'] = owf_revenues_to_electrolyser_list
    df_cost_data.loc['revenues_to_market'] = owf_revenues_to_market_list

    df_cost_data

    # the scenario is chosen, by dropping the other two scenario's from the dataframe

    df_cost_data.drop(columns=drop_scenarios,axis=1,inplace=True)
    df_cost_data.style.format(precision=3)

    # get cost data for the correct scenario
    # in the df we can see that we have the most_likely scenario
    # we can get the parameters from the df by using the row index
    # using df_cost_data.loc['parameter'] not only gives the value, but also column name, dtype etc.
    # therefore, we use .item() to get the desired value

    duration_construction = int(df_cost_data.loc['duration_construction'].item())
    duration_operation = int(df_cost_data.loc['duration_operation'].item())
    duration_decommissioning = int(df_cost_data.loc['duration_decommissioning'].item())
    tender_year = int(df_cost_data.loc['tender_year'].item())

    owf_income_tax_rate = df_cost_data.loc['income_tax_rate'].item()
    owf_inflation = df_cost_data.loc['inflation'].item()                 
    # owf_general_WACC = df_cost_data.loc['wacc'].item()
    owf_loan_interest_rate = df_cost_data.loc['loan_interest_rate'].item()
    owf_length_of_loan = int(df_cost_data.loc['length_of_loan'].item())
    owf_depreciation = int(df_cost_data.loc['depreciation'].item())

    owf_contingency = df_cost_data.loc['contingency'].item()
    owf_loan_percentage = df_cost_data.loc['loan_percentage'].item()
    owf_decomissioning_percentage = df_cost_data.loc['decommissioning_percentage'].item()

    owf_sold_to_electrolyser = df_cost_data.loc['sold_to_electrolyser'].item()
    owf_sold_to_grid = df_cost_data.loc['sold_to_grid'].item()
    owf_revenues_to_electrolyser = df_cost_data.loc['revenues_to_electrolyser'].item()
    owf_revenues_to_market = df_cost_data.loc['revenues_to_market'].item()


    ########## Calculate cost data

    # We need the number of turbines because some of the cost data is given per turbine
    # We assume that at least 700 MW windfarm capacity is required
    # Therefore we use np.ceil (or you could use math.ceil) to round the value up to the closest integer
    # Then we multiply the rounded number with the turbine capacity to get a new total windfarm capacity,
    # which is equal to or slightly higher than the original 700 MW capacity

    number_of_turbines = np.ceil(windfarm_capacity / df_yearly_data[tender_year]['windturbine_capacity'])
    new_windfarm_capaxity = number_of_turbines * df_yearly_data[tender_year]['windturbine_capacity']

    # The cable costs depend on the length of the cables
    # In Mapeditor, a straight line between OWF and Eemshaven is approximately 111694 meter
    cable_distance = 111694 / 1000     # km

    owf_capex = (df_yearly_data[tender_year]['capex_rna'] * number_of_turbines +
                    df_yearly_data[tender_year]['capex_structure'] * number_of_turbines +
                    df_yearly_data[tender_year]['capex_electric'] * new_windfarm_capaxity / 1000 +
                    df_yearly_data[tender_year]['capex_cables'] * new_windfarm_capaxity / 1000 * cable_distance +
                    df_yearly_data[tender_year]['capex_installation'] * new_windfarm_capaxity / 1000 +
                    df_yearly_data[tender_year]['capex_project_costs'] * new_windfarm_capaxity / 1000)




    ##################################### END OF ALL THE INPUTS THAT DEPEND ON THE SCENARIO    ################################


    owf_run_all_functions(owf_capex,
                        df_owf_opex,
                        owf_inflation,
                        owf_revenues_to_electrolyser,
                        owf_revenues_to_market,
                        owf_loan_percentage,
                        owf_loan_interest_rate,
                        owf_income_tax_rate,
                        owf_general_WACC,
                        duration_operation)

    # create a temporary column of dummy data at the top-level of the MultiIndex (which will later be dropped)
    # without first creating this temporary column, pandas will not be able to assign values to the MultiIndex df
    df_lcr_all_scenarios.loc[:, 'temp'] = levelized_cost_and_revenues(owf_capex,
                                                                                        df_owf_opex,
                                                                                        owf_inflation,
                                                                                        owf_revenues_to_electrolyser,
                                                                                        owf_revenues_to_market,
                                                                                        owf_loan_percentage,
                                                                                        owf_loan_interest_rate,
                                                                                        owf_income_tax_rate,
                                                                                        owf_general_WACC,
                                                                                        duration_operation)['cost']
    if x == file_names[0]:
        # add to pessimistic
        df_lcr_all_scenarios.loc[:, ('pessimistic','costs')] = levelized_cost_and_revenues(owf_capex,
                                                                                        df_owf_opex,
                                                                                        owf_inflation,
                                                                                        owf_revenues_to_electrolyser,
                                                                                        owf_revenues_to_market,
                                                                                        owf_loan_percentage,
                                                                                        owf_loan_interest_rate,
                                                                                        owf_income_tax_rate,
                                                                                        owf_general_WACC,
                                                                                        duration_operation)['cost']
        
        df_lcr_all_scenarios.loc[:, ('pessimistic','revenues')] = levelized_cost_and_revenues(owf_capex,
                                                                                        df_owf_opex,
                                                                                        owf_inflation,
                                                                                        owf_revenues_to_electrolyser,
                                                                                        owf_revenues_to_market,
                                                                                        owf_loan_percentage,
                                                                                        owf_loan_interest_rate,
                                                                                        owf_income_tax_rate,
                                                                                        owf_general_WACC,
                                                                                        duration_operation)['revenues'] 

    elif x == file_names[1]:
        # add to most_likely
        df_lcr_all_scenarios.loc[:, ('most_likely','costs')] = levelized_cost_and_revenues(owf_capex,
                                                                                        df_owf_opex,
                                                                                        owf_inflation,
                                                                                        owf_revenues_to_electrolyser,
                                                                                        owf_revenues_to_market,
                                                                                        owf_loan_percentage,
                                                                                        owf_loan_interest_rate,
                                                                                        owf_income_tax_rate,
                                                                                        owf_general_WACC,
                                                                                        duration_operation)['cost']
        
        df_lcr_all_scenarios.loc[:, ('most_likely','revenues')] = levelized_cost_and_revenues(owf_capex,
                                                                                        df_owf_opex,
                                                                                        owf_inflation,
                                                                                        owf_revenues_to_electrolyser,
                                                                                        owf_revenues_to_market,
                                                                                        owf_loan_percentage,
                                                                                        owf_loan_interest_rate,
                                                                                        owf_income_tax_rate,
                                                                                        owf_general_WACC,
                                                                                        duration_operation)['revenues']
    else: 
        # add to optimistic
        df_lcr_all_scenarios.loc[:, ('optimistic','costs')] = levelized_cost_and_revenues(owf_capex,
                                                                                        df_owf_opex,
                                                                                        owf_inflation,
                                                                                        owf_revenues_to_electrolyser,
                                                                                        owf_revenues_to_market,
                                                                                        owf_loan_percentage,
                                                                                        owf_loan_interest_rate,
                                                                                        owf_income_tax_rate,
                                                                                        owf_general_WACC,
                                                                                        duration_operation)['cost']
        
        df_lcr_all_scenarios.loc[:, ('optimistic','revenues')] = levelized_cost_and_revenues(owf_capex,
                                                                                        df_owf_opex,
                                                                                        owf_inflation,
                                                                                        owf_revenues_to_electrolyser,
                                                                                        owf_revenues_to_market,
                                                                                        owf_loan_percentage,
                                                                                        owf_loan_interest_rate,
                                                                                        owf_income_tax_rate,
                                                                                        owf_general_WACC,
                                                                                        duration_operation)['revenues']
# drop temporary column that was only used for correct multi-level dataframe assignment        

df_lcr_all_scenarios = df_lcr_all_scenarios.drop(columns='temp')



XMLResource loading...
XMLResource loading...
XMLResource loading...


/var/folders/ft/zfpy89bj74s_ytd4g9pc9dz80000gn/T/ipykernel_70828/323911502.py:342: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df_lcr_all_scenarios = df_lcr_all_scenarios.drop(columns='temp')


In [ ]:
df_lcr_all_scenarios

Scenario         pessimistic             most_likely            optimistic  \
Type                   costs    revenues       costs   revenues      costs   
capex              95.184337         0.0   67.235476        0.0  47.313413   
opex               29.422552         0.0   17.401319        0.0  12.874019   
decommissioning     0.517352         0.0    0.517352        0.0    0.32765   
interest_costs     41.995343         0.0   13.246231        0.0   4.503246   
contingency        10.213202         0.0    7.214312        0.0   5.237746   
tax_expenses            -0.0         0.0    3.776829        0.0  13.390164   
revenues_ppa             0.0   50.068777         0.0  66.759771        0.0   
revenues_market          0.0    1.635053         0.0   2.181472        0.0   
profits                  0.0         0.0         0.0        0.0   4.334805   
unprofitable_gap         0.0  125.628956         0.0  40.450276        0.0   

Scenario                     
Type               revenues  
capex                   0.0  
opex                    0.0  
decommissioning         0.0  
interest_costs          0.0  
contingency             0.0  
tax_expenses            0.0  
revenues_ppa      85.200255  
revenues_market    2.780788  
profits                 0.0  
unprofitable_gap        0.0